In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from Log_Extractor import LogExtractor
from pump_config import PUMP_TO_TANK
import joblib
from sklearn.ensemble import RandomForestRegressor
from Ana_Preprocess import VirtualSensorPreprocessor
import os

In [2]:
def prepare_pump_features(raw_df, pump_id, tank_id, robot_id):
    """
    raw_df를 받아 해당 펌프(pump_id)의 피처를 생성하여 model_df를 반환합니다.
    """
    df = raw_df.ffill().fillna(0).copy()
    df_reset = df.reset_index() 
    
    # 동적 태그 이름 생성 (P1, P2 등 자동 대응)
    tag_SV = f'g_s_SV_{pump_id}'
    tag_Ana = f'Ana_Out_{pump_id}'
    tag_PT = f'Scale_Out___PT_{pump_id}'
    tag_FT = f'Scale_Out___FT_{pump_id}'
    tag_Temp = f'TK_Temp_PV_{tank_id}'

    # [신규] 완벽한 타이밍을 위한 하드 트리거 태그
    tag_BuildUp = f'Pump_BuildUp_{pump_id}' 
    tag_Wagon = f'{robot_id}_Robot_Num'

    # ==========================================
    # ✂️ 1. 완벽한 칼단발 블록(Block) 나누기
    # ==========================================
    # 대차 번호가 바뀌거나, 펌프 가동 상태(ON/OFF)가 바뀔 때마다 새로운 블록 ID를 부여합니다.
    df_reset['Wagon_Changed'] = df_reset[tag_Wagon].diff() != 0
    df_reset['BuildUp_Changed'] = df_reset[tag_BuildUp].diff() != 0
    df_reset['Wagon_Block_ID'] = (df_reset['Wagon_Changed'] | df_reset['BuildUp_Changed']).cumsum()

    # ==========================================
    # 🎯 2. 진짜 샷 추출 (조건부 필터링 싹 다 폐기)
    # ==========================================
    # BuildUp이 1(가동 중)인 데이터만 남깁니다. 이것이 진짜 샷입니다.
    df_shots = df_reset[df_reset[tag_BuildUp] == 1].copy()

    # ==========================================
    # 🔄 3. 이전 SV 값(Prev_SV) 구하기
    # ==========================================
    # 각 유효 블록별로 최대 SV를 구한 뒤, 이전 샷의 SV를 1칸 당겨옵니다.
    block_summary = df_shots.groupby('Wagon_Block_ID').agg(
        Max_SV=(tag_SV, 'max')
    )
    block_summary['Prev_SV'] = block_summary['Max_SV'].shift(1)

    # 원본 df_shots에 Prev_SV 정보 맵핑
    df_shots = df_shots.merge(block_summary[['Prev_SV']], left_on='Wagon_Block_ID', right_index=True, how='left')
    df_shots = df_shots.sort_index()

    # ==========================================
    # 📈 4. 실시간 피처 엔지니어링 (기존 로직 완벽 유지)
    # ==========================================
    model_df = df_shots.copy()
    model_df['Tick_Index'] = model_df.groupby('Wagon_Block_ID').cumcount()
    model_df['Phase_Start'] = np.where(model_df['Tick_Index'] < 2, 1.0, 0.0)
    model_df['Phase_Steady'] = np.where(model_df['Tick_Index'] >= 2, 1.0, 0.0)

    model_df['Instant_FT_Error'] = model_df[tag_SV] - model_df[tag_FT]
    model_df['Instant_FT_Error_Rate'] = np.where(
        model_df[tag_SV] > 0, 
        (model_df['Instant_FT_Error'] / model_df[tag_SV]) * 100, 
        0
    )
    model_df['Cum_FT_Error'] = model_df.groupby('Wagon_Block_ID')['Instant_FT_Error'].cumsum()

    model_df['Prev_SV_Diff'] = model_df[tag_SV] - model_df['Prev_SV'].fillna(0)
    model_df['Rolling_PT_Max_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].rolling(window=3, min_periods=1).max().reset_index(0, drop=True)
    model_df['Rolling_PT_Diff_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].diff(periods=2).fillna(0)

    model_df['Instant_SV_Diff'] = model_df[tag_SV].diff().fillna(0)
    model_df['Phase_Transition'] = np.where(model_df['Instant_SV_Diff'] != 0, 1.0, 0.0)

    # ==========================================
    # 🧹 5. 최종 컬럼 정리
    # ==========================================
    feature_cols = [
        tag_SV, 'Prev_SV', 'Prev_SV_Diff', 
        tag_Ana, tag_Temp,               
        tag_PT, 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
        tag_FT, 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
        'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
    ]
    
    # 결측치 최종 정리 (Shift나 Rolling에서 발생한 NaN 처리)
    model_df = model_df.fillna(0)
    print("✅ 동적 피처 생성 함수 준비 완료! (하드 트리거 기반)")
    
    return model_df[feature_cols], feature_cols


In [3]:
extract = LogExtractor()
start_time="2026-04-13T01:00:00Z"
end_time="2026-04-17T15:00:00Z"

🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!


In [ ]:
pid = "P1"
pump_info = PUMP_TO_TANK.get(pid, {})
tank_id = pump_info.get("Tank", pid)
robot_id = pump_info.get("Robot", None)

target_tags = [
    f'Pump_BuildUp_{pid}' , f'{robot_id}_Robot_Num', 
    f'g_s_SV_{pid}', f'Ana_Out_{pid}', 
    f'Scale_Out___PT_{pid}', f'Scale_Out___FT_{pid}', 
    f'TK_Temp_PV_{tank_id}', f'TK_Level_PV_{tank_id}'
]

raw_df = extract.get_data(start_time=start_time, end_time=end_time, target_tags=target_tags)
train_df, feature_cols = prepare_pump_features(raw_df, pid, tank_id, robot_id)
print(f"✅ {pid} 데이터 준비 완료! 총 {len(train_df)}틱, 피처 수: {len(feature_cols)}")

In [ ]:
# 1. 아까 정의한 '순수 물리 피처' 7개
pure_physical_features = [
    f'g_s_SV_{pid}', f'Prev_SV', f'Prev_SV_Diff', f'Ana_Out_{pid}', f'TK_Temp_PV_{tank_id}',
    f'Scale_Out___FT_{pid}', 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
    'Phase_Start', 'Phase_Steady', 'Phase_Transition'
]

In [ ]:
# X: 순수 물리 피처 7개만 쏙 뽑기
X_train = train_df[pure_physical_features]
# y: 정답지 (Scale_Out___PT_)
y_train = train_df[f'Scale_Out___PT_{pid}']

print(f"📊 학습 데이터 준비 완료: {X_train.shape} (7개 피처만 사용!)")

# 3. 모델 학습 (Random Forest)
virtual_sensor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
virtual_sensor.fit(X_train, y_train)

# 4. 저장 (기존 구형 모델 덮어쓰기)
joblib.dump(virtual_sensor, f'PT_models/virtual_sensor_{pid}.pkl')
print(f"✅ 순수 물리 가상 센서 저장 완료! 이제 아까 그 테스트 코드를 다시 돌려보세요!")

In [ ]:
raw_valid = extract.get_data(start_time="-1d", end_time="now()", target_tags=target_tags)

In [ ]:
import joblib
from Ana_Preprocess import VirtualSensorPreprocessor

print("🚀 가상 센서 (Virtual Sensor) 단독 테스트 시작...\n")

# 1. 학습된 가상 센서(Random Forest) 로드
# (앞서 Ana_Out을 정답지(y)로 두고 학습한 모델이 있다고 가정합니다)
try:
    virtual_pt_sensor = joblib.load(f'PT_models/virtual_sensor_{pid}.pkl')
    print("✅ 가상 PT 센서 모델 로드 완료!")
except FileNotFoundError:
    print("⚠️ 모델 파일이 없습니다. 코드가 정상 작동하는지 흐름만 테스트합니다.")
    virtual_pt_sensor = None


# 전처리기 가동
vs_preprocessor = VirtualSensorPreprocessor(feature_cols=target_tags, pump_id=pid, tank_id=tank_id, robot_id=robot_id)

# 3. 실시간 스트리밍 시뮬레이션 루프
records = raw_valid.to_dict('records')
loss_alerts = []

for i, live_row in enumerate(records):
    # 전처리기에 1틱 입력
    df_processed, meta_info = vs_preprocessor.process_raw_tick(live_row)
    
    # 가동 중이 아니면(BuildUp == 0) 패스
    if df_processed is None:
        continue
        
    timestamp = live_row.get('Time', 'Unknown Time')

    # 🚨 타겟 변경: 실제 측정된 PT 값 추출
    actual_pt = df_processed.iloc[0][f'Scale_Out___PT_{pid}']
    
    # 모델 입력 데이터는 형님이 정한 'PT 예측용 피처'만 칼같이 잘라냅니다!
    X_input = df_processed[pure_physical_features]
    
    if virtual_pt_sensor:
        # 가상 센서 예측: "이 설정(SV)과 전류(Ana)라면, 정상적인 펌프일 때 압력(PT)이 얼마여야 하는가?"
        virtual_pt = virtual_pt_sensor.predict(X_input)[0]
        
        # 압력 손실률 계산 (%)
        # 정상 상태(가상 센서) 대비 실제 압력이 얼마나 떨어졌는가?
        if virtual_pt > 0:
            pressure_loss_rate = ((virtual_pt - actual_pt) / virtual_pt) * 100
        else:
            pressure_loss_rate = 0.0
            
        # 디버깅용 로그 (정상일 때도 출력해보고 싶으시면 주석 해제)
        # print(f"[{timestamp}] [틱 {meta_info['Tick_Index']:02d}] 정상 기대 PT: {virtual_pt:.1f} | 실제 측정 PT: {actual_pt:.1f} | 손실: {pressure_loss_rate:.1f}%")
            
        # 5% 이상 압력이 샌다면(기대치보다 낮다면) 경고 출력
        if pressure_loss_rate > 5.0:
            robot_num = meta_info['Robot_Num']
            tick = meta_info['Tick_Index']
            msg = f"🚨 [로봇 {robot_num}번 / {tick}틱] 펌프 압력 저하 감지! (정상 기대 PT: {virtual_pt:.1f} vs 실제 측정 PT: {actual_pt:.1f} | 손실률: {pressure_loss_rate:.1f}%)"
            print(msg)
            loss_alerts.append(msg)

print("\n" + "="*60)
print(f"✅ 가상 PT 센서 테스트 완료! 총 {len(records)}틱 중 {len(loss_alerts)}회의 압력 손실 알람 발생.")
print("="*60)